# Minimal CUDA reproducer: Triad isopycnal diffusion + TEOS-10

This notebook tests a device-side `DomainError` observed on the first timestep with `TriadIsopycnalSkewSymmetricDiffusivity`. It intentionally removes the Amazon grid, NumericalEarth, rivers, dye, WENO, free surface, forcing, sponges, output, and plotting.

The suspected minimal interaction is **CUDA + Triad + nonlinear TEOS-10 buoyancy**. Run each GPU case in a fresh Julia kernel if the previous one throws a device exception.

In [1]:
ENV["CUDA_LAUNCH_BLOCKING"] = "1"

using Pkg
using Oceananigans
using CUDA
using Oceananigans.BuoyancyFormulations.SeawaterPolynomials.TEOS10: TEOS10EquationOfState
using Oceananigans.TurbulenceClosures: TriadIsopycnalSkewSymmetricDiffusivity

CUDA.allowscalar(false)
println("Julia: ", VERSION)
Pkg.status(["Oceananigans", "SeawaterPolynomials", "CUDA", "CUDACore"])
println("CUDA.functional() = ", CUDA.functional())

Julia: 1.12.6
Status `C:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\repo-cleanup\ocean-modeling\amazon_river\Project.toml` (empty project)
CUDA.functional() = true


## Tiny model

The model is an 8 x 8 x 8 closed box with motion initially zero. Temperature and salinity are smooth, finite, and within normal oceanographic ranges. A small horizontal and vertical density gradient gives Triad nonzero slopes. There is no passive dye because dye is not involved in the reported `DomainError`.

In [2]:
const N = 8
const L = 80_000.0
const H = 800.0
const dt = 1.0

function build_model(arch; triad=true, nonlinear_eos=true, immersed=false)
    base_grid = RectilinearGrid(arch;
        size=(N, N, N),
        halo=(2, 2, 2),
        x=(0, L), y=(0, L), z=(-H, 0),
        topology=(Bounded, Bounded, Bounded))

    if immersed
        bottom(x, y) = -700.0 + 250.0 * exp(-((x-L/2)^2 + (y-L/2)^2) / (15_000.0)^2)
        grid = ImmersedBoundaryGrid(base_grid, GridFittedBottom(bottom); active_cells_map=true)
    else
        grid = base_grid
    end

    eos = TEOS10EquationOfState()
    buoyancy = nonlinear_eos ? SeawaterBuoyancy(equation_of_state=eos) : SeawaterBuoyancy()
    closure = triad ? TriadIsopycnalSkewSymmetricDiffusivity(
        κ_skew=1e3, κ_symmetric=1e3) : nothing

    model = NonhydrostaticModel(grid;
        tracers=(:T, :S),
        buoyancy, closure,
        advection=nothing)

    T0(x, y, z) = 18.0 + 2e-5 * (x-L/2) + 0.008 * z
    S0(x, y, z) = 35.0 + 1e-5 * (y-L/2) - 2e-4 * z
    set!(model, T=T0, S=S0)
    Oceananigans.BoundaryConditions.fill_halo_regions!(model.tracers)
    return model
end

build_model (generic function with 1 method)

## Diagnostics and one-step runner

Both active interior values and the complete parent arrays (including halos) are checked before the timestep. TEOS-10 evaluates a square root involving `S + 32`; therefore the report explicitly counts values below `-32`.

In [3]:
function tracer_report(model, label)
    Tint = Array(interior(model.tracers.T))
    Sint = Array(interior(model.tracers.S))
    Tall = Array(parent(model.tracers.T))
    Sall = Array(parent(model.tracers.S))
    println(label)
    println("  interior T range: ", extrema(Tint))
    println("  interior S range: ", extrema(Sint))
    println("  all-storage T range: ", extrema(Tall))
    println("  all-storage S range: ", extrema(Sall))
    println("  all-storage S < -32 count: ", count(<(-32), Sall))
    @assert all(isfinite, Tint) && all(isfinite, Sint)
    @assert minimum(Sint) > -32
    return nothing
end

function run_one_step(name, arch; triad=true, nonlinear_eos=true, immersed=false)
    println("\n=== ", name, " ===")
    model = build_model(arch; triad, nonlinear_eos, immersed)
    tracer_report(model, "before timestep")
    time_step!(model, dt)
    arch isa GPU && CUDA.synchronize()
    tracer_report(model, "after timestep")
    println("PASS: ", name)
    return model
end

run_one_step (generic function with 1 method)

## Case 1: CPU + Triad + TEOS-10

This is the CPU reference. It should complete one timestep.

In [4]:
cpu_triad_teos = run_one_step("CPU + Triad + TEOS-10 + regular grid", CPU())


=== CPU + Triad + TEOS-10 + regular grid ===
before timestep
  interior T range: (11.3, 18.3)
  interior S range: (34.66, 35.5)
  all-storage T range: (0.0, 18.3)
  all-storage S range: (0.0, 35.5)
  all-storage S < -32 count: 0
after timestep
  interior T range: (11.299999096424447, 18.30000087083264)
  interior S range: (34.66000028118662, 35.499999756700504)
  all-storage T range: (0.0, 18.30000087083264)
  all-storage S range: (0.0, 35.499999756700504)
  all-storage S < -32 count: 0
PASS: CPU + Triad + TEOS-10 + regular grid


NonhydrostaticModel{CPU, RectilinearGrid}(time = 1 second, iteration = 1)
├── grid: 8×8×8 RectilinearGrid{Float64, Bounded, Bounded, Bounded} on CPU with 2×2×2 halo
├── timestepper: RungeKutta3TimeStepper
├── advection scheme: Nothing
├── tracers: (T, S)
├── closure: TriadIsopycnalSkewSymmetricDiffusivity(κ_skew=1000.0, κ_symmetric=1000.0)
├── buoyancy: SeawaterBuoyancy with g=9.80665 and BoussinesqEquationOfState{Float64} with ĝ = NegativeZDirection()
└── coriolis: Nothing

## Case 2: GPU + TEOS-10 without Triad

This controls for CUDA and TEOS-10 without the Triad closure. Run in a fresh kernel after any prior CUDA device exception.

In [5]:
@assert CUDA.functional()
gpu_teos_control = run_one_step("GPU + no Triad + TEOS-10 + regular grid", GPU(); triad=false)


=== GPU + no Triad + TEOS-10 + regular grid ===
before timestep
  interior T range: (11.3, 18.3)
  interior S range: (34.66, 35.5)
  all-storage T range: (0.0, 18.3)
  all-storage S range: (0.0, 35.5)
  all-storage S < -32 count: 0
after timestep
  interior T range: (11.3, 18.3)
  interior S range: (34.66, 35.5)
  all-storage T range: (0.0, 18.3)
  all-storage S range: (0.0, 35.5)
  all-storage S < -32 count: 0
PASS: GPU + no Triad + TEOS-10 + regular grid


NonhydrostaticModel{CUDAGPU, RectilinearGrid}(time = 1 second, iteration = 1)
├── grid: 8×8×8 RectilinearGrid{Float64, Bounded, Bounded, Bounded} on CUDAGPU with 2×2×2 halo
├── timestepper: RungeKutta3TimeStepper
├── advection scheme: Nothing
├── tracers: (T, S)
├── closure: Nothing
├── buoyancy: SeawaterBuoyancy with g=9.80665 and BoussinesqEquationOfState{Float64} with ĝ = NegativeZDirection()
└── coriolis: Nothing

## Case 3: GPU + Triad with linear buoyancy

This controls for the Triad GPU kernels while removing the nonlinear TEOS-10 calculation.

In [6]:
gpu_triad_linear = run_one_step("GPU + Triad + linear EOS + regular grid", GPU(); nonlinear_eos=false)


=== GPU + Triad + linear EOS + regular grid ===
before timestep
  interior T range: (11.3, 18.3)
  interior S range: (34.66, 35.5)
  all-storage T range: (0.0, 18.3)
  all-storage S range: (0.0, 35.5)
  all-storage S < -32 count: 0
after timestep
  interior T range: (11.299999083234042, 18.30000091676596)
  interior S range: (34.66000019628194, 35.499999803718055)
  all-storage T range: (0.0, 18.30000091676596)
  all-storage S range: (0.0, 35.499999803718055)
  all-storage S < -32 count: 0
PASS: GPU + Triad + linear EOS + regular grid


NonhydrostaticModel{CUDAGPU, RectilinearGrid}(time = 1 second, iteration = 1)
├── grid: 8×8×8 RectilinearGrid{Float64, Bounded, Bounded, Bounded} on CUDAGPU with 2×2×2 halo
├── timestepper: RungeKutta3TimeStepper
├── advection scheme: Nothing
├── tracers: (T, S)
├── closure: TriadIsopycnalSkewSymmetricDiffusivity(κ_skew=1000.0, κ_symmetric=1000.0)
├── buoyancy: SeawaterBuoyancy with g=9.80665 and LinearEquationOfState(thermal_expansion=0.000167, haline_contraction=0.00078) with ĝ = NegativeZDirection()
└── coriolis: Nothing

## Case 4: suspected minimal failure

This is the key case: GPU + Triad + TEOS-10 on a regular grid. If it fails while cases 1-3 pass, the minimal interaction does not require immersed boundaries or any Amazon-model component. Save the entire output and restart Julia after a device exception.

In [7]:
gpu_triad_teos = run_one_step("GPU + Triad + TEOS-10 + regular grid", GPU())


=== GPU + Triad + TEOS-10 + regular grid ===
before timestep
  interior T range: (11.3, 18.3)
  interior S range: (34.66, 35.5)
  all-storage T range: (0.0, 18.3)
  all-storage S range: (0.0, 35.5)
  all-storage S < -32 count: 0
after timestep
  interior T range: (11.299999096424447, 18.30000087083264)
  interior S range: (34.66000028118662, 35.499999756700504)
  all-storage T range: (0.0, 18.30000087083264)
  all-storage S range: (0.0, 35.499999756700504)
  all-storage S < -32 count: 0
PASS: GPU + Triad + TEOS-10 + regular grid


NonhydrostaticModel{CUDAGPU, RectilinearGrid}(time = 1 second, iteration = 1)
├── grid: 8×8×8 RectilinearGrid{Float64, Bounded, Bounded, Bounded} on CUDAGPU with 2×2×2 halo
├── timestepper: RungeKutta3TimeStepper
├── advection scheme: Nothing
├── tracers: (T, S)
├── closure: TriadIsopycnalSkewSymmetricDiffusivity(κ_skew=1000.0, κ_symmetric=1000.0)
├── buoyancy: SeawaterBuoyancy with g=9.80665 and BoussinesqEquationOfState{Float64} with ĝ = NegativeZDirection()
└── coriolis: Nothing

## Case 5: add an immersed bottom only if Case 4 passes

If Case 4 passes, restart Julia and run the setup cells followed by this cell. A failure here would isolate the additional requirement to the interaction with an immersed boundary.

In [8]:
gpu_triad_teos_immersed = run_one_step("GPU + Triad + TEOS-10 + immersed grid", GPU(); immersed=true)


=== GPU + Triad + TEOS-10 + immersed grid ===


┌ Warning: The FFT-based pressure_solver for NonhydrostaticModels on ImmersedBoundaryGrid
│ is approximate and will probably produce velocity fields that are divergent
│ adjacent to the immersed boundary. An experimental but improved pressure_solver
│ is available which may be used by writing
│ 
│     using Oceananigans.Solvers: ConjugateGradientPoissonSolver
│     pressure_solver = ConjugateGradientPoissonSolver(grid)
│ 
│ Please report issues to https://github.com/CliMA/Oceananigans.jl/issues.
└ @ Oceananigans.Models.NonhydrostaticModels C:\Users\meghn\.julia\packages\Oceananigans\NCFoc\src\Models\NonhydrostaticModels\NonhydrostaticModels.jl:69


before timestep
  interior T range: (0.0, 18.3)
  interior S range: (0.0, 35.480000000000004)
  all-storage T range: (0.0, 18.3)
  all-storage S range: (0.0, 35.480000000000004)
  all-storage S < -32 count: 0
after timestep
  interior T range: (0.0, 18.30000087083264)
  interior S range: (0.0, 35.47999975116085)
  all-storage T range: (0.0, 18.30000087083264)
  all-storage S range: (0.0, 35.47999975116085)
  all-storage S < -32 count: 0
PASS: GPU + Triad + TEOS-10 + immersed grid


NonhydrostaticModel{CUDAGPU, ImmersedBoundaryGrid}(time = 1 second, iteration = 1)
├── grid: 8×8×8 ImmersedBoundaryGrid{Float64, Bounded, Bounded, Bounded} on CUDAGPU with 2×2×2 halo
├── timestepper: RungeKutta3TimeStepper
├── advection scheme: Nothing
├── tracers: (T, S)
├── closure: TriadIsopycnalSkewSymmetricDiffusivity(κ_skew=1000.0, κ_symmetric=1000.0)
├── buoyancy: SeawaterBuoyancy with g=9.80665 and BoussinesqEquationOfState{Float64} with ĝ = NegativeZDirection()
└── coriolis: Nothing

## Reporting the result

For an Oceananigans issue, include: package versions, GPU model, the first failing case, all earlier passing cases, pre-step tracer diagnostics, and the full device exception. Do not describe this as a passive-dye failure: this reproducer specifically tests the Triad/TEOS-10 CUDA path. If all five cases pass, add one omitted Amazon feature at a time; this notebook is then not yet a reproducer.